# FREE DAILY VIDEO ENGINE — Kobe (animal) + Brendan (human)

**$0. One Colab T4 session produces both clips.** Avoids HF's ~300s/day ZeroGPU cap
(LatentSync Spaces reserve 180s *per call*).

| Phase | Model | Who | Note |
|---|---|---|---|
| A | **JoyVASA** `--animation_mode animal` | 🐶 Kobe | Only free model trained for **animal** faces |
| B | **LatentSync** | 🧑 Brendan | Best open lip-sync for humans |

**Runtime → Change runtime type → T4 GPU → Run all.**

*Commands verified 2026-09-05 against both repos (JoyVASA: `--reference/--audio/--animation_mode`;
LatentSync: `--unet_config_path/--inference_ckpt_path/--video_path/--audio_path/--video_out_path`).*

### Before running: upload 4 files to the Colab Files panel
- `kobe.png` — clear front-facing Kobe (`Brands/Kobe/refs/kobe-viewsai.png`)
- `kobe.wav` — Kobe's 8–10s line (Kokoro TTS → wav)
- `brendan.mp4` — short clip of Brendan (skip Phase B if not needed)
- `brendan.wav` — Brendan's line

## 0. Setup + upload

In [ ]:
!apt-get install -y ffmpeg libgl1 git-lfs > /dev/null 2>&1
!git lfs install --skip-repo > /dev/null 2>&1
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
print('GPU ready')

In [ ]:
from google.colab import files
print('Upload: kobe.png, kobe.wav (and brendan.mp4, brendan.wav for Phase B)')
files.upload()

# PHASE A — KOBE (JoyVASA, animal mode) 🐶
Audio → identity-independent facial motion (diffusion) → rendered via LivePortrait.
`--animation_mode animal` is what makes a Pomeranian snout work where human-face models fail.

In [ ]:
%cd /content
!git clone -q https://github.com/jdh-algo/JoyVASA.git
%cd /content/JoyVASA
!pip install -q tyro==0.8.5 accelerate==0.28.0 bitsandbytes==0.43.1 diffusers==0.27.2 \
    einops==0.8.0 librosa==0.10.2.post1 mediapipe==0.10.14 imageio-ffmpeg pykalman \
    opencv-python scipy scikit-image onnxruntime-gpu soundfile
print('JoyVASA deps installed')

In [ ]:
import os
os.makedirs('/content/JoyVASA/pretrained_weights', exist_ok=True)
%cd /content/JoyVASA
!git clone -q https://huggingface.co/jdh-algo/JoyVASA pretrained_weights/JoyVASA
!git clone -q https://huggingface.co/facebook/wav2vec2-base-960h pretrained_weights/wav2vec2-base-960h
!git clone -q https://huggingface.co/KwaiVGI/LivePortrait pretrained_weights/liveportrait
!ls pretrained_weights
print('checkpoints ready')

In [ ]:
# ---- KOBE INFERENCE (animal mode) ----
%cd /content/JoyVASA
!python inference.py \
  --reference /content/kobe.png \
  --audio /content/kobe.wav \
  --animation_mode animal \
  --output_dir /content/outputs/kobe \
  --flag_stitching False

import glob, shutil, os
vids = sorted(glob.glob('/content/outputs/kobe/**/*.mp4', recursive=True), key=os.path.getmtime)
if vids:
    shutil.copy(vids[-1], '/content/kobe_talking.mp4')
    print('KOBE CLIP:', vids[-1])
    from google.colab import files as f2
    f2.download('/content/kobe_talking.mp4')
else:
    print('no output - see log above')

### If Kobe/JoyVASA errors
Try each one-line change in order:
1. `--flag_use_half_precision True` (T4 fp16, faster)
2. `--animation_mode human` (some builds need the human renderer to init first)
3. `--flag_pasteback True` (return the full original frame instead of the face crop)

# PHASE B — BRENDAN (LatentSync) 🧑
No HF ZeroGPU quota here — runs on the Colab T4 directly.

*Note: the repo's own `setup_env.sh` uses **conda** (not available on Colab), so we install manually below.*

In [ ]:
%cd /content
!git clone -q https://github.com/bytedance/LatentSync.git
%cd /content/LatentSync
!pip install -q -r requirements.txt 2>&1 | tail -3
print('LatentSync deps installed')

In [ ]:
# Checkpoints: unet (5GB) + whisper/tiny.pt — exact paths from the repo's setup_env.sh
%cd /content/LatentSync
!huggingface-cli download ByteDance/LatentSync-1.6 whisper/tiny.pt --local-dir checkpoints
!huggingface-cli download ByteDance/LatentSync-1.6 latentsync_unet.pt --local-dir checkpoints
!ls -R checkpoints | head -10
print('checkpoints ready')

In [ ]:
# ---- BRENDAN INFERENCE ----
%cd /content/LatentSync
!python -m scripts.inference \
  --unet_config_path configs/unet/second_stage.yaml \
  --inference_ckpt_path checkpoints/latentsync_unet.pt \
  --inference_steps 20 \
  --guidance_scale 1.5 \
  --video_path /content/brendan.mp4 \
  --audio_path /content/brendan.wav \
  --video_out_path /content/brendan_out.mp4

import os
if os.path.exists('/content/brendan_out.mp4'):
    print('BRENDAN CLIP ready')
    from google.colab import files as f3
    f3.download('/content/brendan_out.mp4')
else:
    print('no output - see log above')

---
# Finish on your Mac (free, local)
```bash
python3 ~/ViewsOSComplete/scripts/free-stack-videos/assemble_kobe.py \
  --movement kobe-fullbody-desk-take3 \
  --line "Drop AI in the comments and I'll send you the whole system" \
  --title "AI recruiting. 24/7." --cta "Comment AI"
```
→ Kokoro voice + safe-zone text → `platform_render.py` → 12 platform-ready files.

**Daily loop:** Run all (~8-10 min) → download 2 clips → assemble → publish.

**Quota reality:** Colab free T4 is ~a few hours/day and sessions can be preempted —
not unlimited, but comfortably enough for 1-3 clips/day.